In [1]:
import os
import math
import pprint
import re
from dotenv import load_dotenv
from datetime import date, timedelta, datetime
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth

load_dotenv()

pd.options.display.max_columns = None  # type: ignore
pd.options.display.float_format = "{:,.2f}".format

token = os.getenv("PERSONAL_ACCESS_TOKEN")

In [2]:
# CONFIG
org = "Dtek-SP"
project = "MarketRMS"
pat = token
auth=HTTPBasicAuth('', pat)
base = f"https://analytics.dev.azure.com/{org}/{project}/_odata/v3.0-preview"
base_url = f"https://dev.azure.com/{org}"
snapshot_path = "work_item_snapshot.parquet"
today = date.today()

# Download live snapshot

In [3]:
# --- 1️⃣ WIQL: filter only this Team Project ---
wiql_query = f"""
SELECT [System.Id]
FROM workitems
WHERE [System.TeamProject] = '{project}'
"""

wiql_url = f"https://dev.azure.com/{org}/{project}/_apis/wit/wiql?api-version=7.0"
resp = requests.post(wiql_url, json={"query": wiql_query}, auth=auth)
resp.raise_for_status()

ids = [w["id"] for w in resp.json().get("workItems", [])]
print(f"✅ Retrieved {len(ids)} work item IDs for project '{project}'")

if not ids:
    raise SystemExit("No work items found for this WIQL query.")

# --- 2️⃣ Fetch work items (fields only, in chunks) ---
wi_batch_url = f"https://dev.azure.com/{org}/_apis/wit/workitemsbatch?api-version=7.0"

# List of fields to fetch
fields_to_get = [
    "System.Id",
    "System.AreaPath",
    "System.IterationPath",
    "System.WorkItemType",
    "System.State",
    "System.AssignedTo",
    "System.CreatedDate",
    "System.CreatedBy",
    "System.ChangedDate",
    "System.ChangedBy",
    "System.Title",
    "Microsoft.VSTS.Scheduling.OriginalEstimate",
    "Microsoft.VSTS.Common.StateChangeDate",
    "Microsoft.VSTS.Common.ClosedDate",
    "Microsoft.VSTS.Common.Priority",
    "System.Tags",
    "Microsoft.VSTS.Scheduling.RemainingWork",
    "System.History",
    "Microsoft.VSTS.Scheduling.CompletedWork",
    "System.TeamProject",
    "System.Parent"
]

rows = []
chunk_size = 200

for i in range(0, len(ids), chunk_size):
    chunk = ids[i:i+chunk_size]
    print(f"🔹 Fetching items {i+1}-{i+len(chunk)} of {len(ids)}")

    body = {
        "ids": chunk,
        "fields": fields_to_get,
        "errorPolicy": "Omit"
    }

    r = requests.post(wi_batch_url, json=body, auth=auth)
    r.raise_for_status()
    rows.extend(r.json().get("value", []))

print(f"✅ Retrieved total {len(rows)} work items")

# --- 3️⃣ Flatten into DataFrame ---
wi_live = pd.json_normalize(rows, sep='.')

# --- 4️⃣ Clean up and rename key columns for readability ---
rename_map = {
    "fields.System.Id": "WorkItemId",
    "fields.System.AreaPath": "AreaPath",
    "fields.System.IterationPath": "IterationPath",
    "fields.System.WorkItemType": "WorkItemType",
    "fields.System.State": "State",
    "fields.System.TeamProject": "ProjectName",
    "fields.System.Parent": "ParentWorkItemId",
    "fields.System.Title": "Title",
    "fields.System.CreatedDate": "CreatedDate",
    "fields.System.ChangedDate": "ChangedDate",
    "fields.Microsoft.VSTS.Scheduling.OriginalEstimate": "OriginalEstimate",
    "fields.Microsoft.VSTS.Common.StateChangeDate": "StateChangeDate",
    "fields.Microsoft.VSTS.Common.ClosedDate": "ClosedDate",
    "fields.Microsoft.VSTS.Common.Priority": "Priority",
    "fields.System.Tags": "TagNames",
    "fields.Microsoft.VSTS.Scheduling.RemainingWork": "RemainingWork",
    "fields.Microsoft.VSTS.Scheduling.CompletedWork": "CompletedWork",
    "fields.System.History": "Comments",
    "fields.System.AssignedTo.displayName": "AssignedTo",
    "fields.System.AssignedTo.uniqueName": "AssignedToEmail",
    "fields.System.CreatedBy.displayName": "CreatedBy",
    "fields.System.CreatedBy.uniqueName": "CreatedByEmail",
    "fields.System.ChangedBy.displayName": "ChangedBy",
    "fields.System.ChangedBy.uniqueName": "ChangedByEmail",
}

wi_live = wi_live.rename(columns=rename_map)

final_cols = [
    "WorkItemId",
    "AreaPath",
    "IterationPath",
    "WorkItemType",
    "State",
    "ProjectName",
    "ParentWorkItemId",
    "Title",
    "CreatedDate",
    "ChangedDate",
    "OriginalEstimate",
    "StateChangeDate",
    "ClosedDate",
    "Priority",
    "TagNames",
    "RemainingWork",
    "CompletedWork",
    "Comments",
    "AssignedTo",
    "AssignedToEmail",
    "CreatedBy",
    "CreatedByEmail",
    "ChangedBy",
    "ChangedByEmail",
]

wi_live = wi_live[final_cols].copy()

✅ Retrieved 1917 work item IDs for project 'MarketRMS'
🔹 Fetching items 1-200 of 1917
🔹 Fetching items 201-400 of 1917
🔹 Fetching items 401-600 of 1917
🔹 Fetching items 601-800 of 1917
🔹 Fetching items 801-1000 of 1917
🔹 Fetching items 1001-1200 of 1917
🔹 Fetching items 1201-1400 of 1917
🔹 Fetching items 1401-1600 of 1917
🔹 Fetching items 1601-1800 of 1917
🔹 Fetching items 1801-1917 of 1917
✅ Retrieved total 1917 work items


In [4]:
def last_segment(series):
    return (
        series.fillna("")
        .str.split("\\")
        .str[-1]
        .replace("", pd.NA)
    )

wi_live["AreaName"] = last_segment(wi_live["AreaPath"])
wi_live["IterationName"] = last_segment(wi_live["IterationPath"])

wi_live["WorkItemURL"] = (
    base_url
    + "/"
    + wi_live["ProjectName"].astype(str)
    + "/_workitems/edit/"
    + wi_live["WorkItemId"].astype(str)
)

ado_dates = [
    "CreatedDate",
    "ChangedDate",
    "StateChangeDate",
    "ClosedDate",
]

for col in ado_dates:
    wi_live[col] = pd.to_datetime(
        wi_live[col],
        utc=True,
        errors="coerce",     # bad/empty values -> NaT instead of crash
        format="ISO8601"     # handles with/without milliseconds
    )

In [5]:
# merge iteration dates

iters_url = f"https://dev.azure.com/{org}/{project}/_apis/wit/classificationnodes/iterations?$depth=10&api-version=7.0"

resp = requests.get(iters_url, auth=auth)
resp.raise_for_status()
tree = resp.json()

def flatten_iterations(node, parent_path=None):
    rows = []
    name = node["name"]
    path = name if parent_path is None else f"{parent_path}\\{name}"

    attrs = node.get("attributes", {})
    rows.append({
        "IterationPath": path,
        "IterationStart": attrs.get("startDate"),
        "IterationEnd": attrs.get("finishDate"),
    })

    for child in node.get("children", []):
        rows.extend(flatten_iterations(child, path))

    return rows

iter_rows = flatten_iterations(tree)
iters_rest_df = pd.DataFrame(iter_rows)

# Convert to datetime if you like
iters_rest_df["IterationStart"] = pd.to_datetime(iters_rest_df["IterationStart"], errors="coerce")
iters_rest_df["IterationEnd"]   = pd.to_datetime(iters_rest_df["IterationEnd"], errors="coerce")

wi_live = wi_live.merge(
    iters_rest_df,
    on="IterationPath",
    how="left"
)

In [6]:
# --- Folder and filename setup ---
output_folder = "live_snapshots"
os.makedirs(output_folder, exist_ok=True)

# today's date in YYYY-MM-DD format
today_str = datetime.today().strftime("%Y-%m-%d")

# file path
file_path = os.path.join(output_folder, f"live_snapshot_{today_str}.parquet")

# --- Save parquet (overwrite if exists) ---
wi_live.to_parquet(file_path, index=False)

print(f"✅ Saved {len(wi_live)} rows to {file_path}")

✅ Saved 1917 rows to live_snapshots\live_snapshot_2026-03-30.parquet


# Download Shapshots history

In [7]:
# Request for new work items

# identify dates to query

existing_snapshots = pd.read_parquet(snapshot_path)
existing_snapshots["SnapshotDate"] = pd.to_datetime(existing_snapshots["SnapshotDate"], utc=True, errors="coerce")
last_snapshot_date = existing_snapshots["SnapshotDate"].max().date()


start = last_snapshot_date + timedelta(days=1)
# ADO Analytics contains data for today. But it may contain yesterday's data till today is actually ended. So we query only for yesterday.
end   = today - timedelta(days=1)

if start > end:
    print("No new days to load – data is up to date.")
    raise SystemExit
else:
    print(f"Will load snapshots from {start} to {end}")

# make a query

wi_url = f"{base}/WorkItemSnapshot"

all_new_rows = []

cur = start
while cur <= end:
    next_day = cur + timedelta(days=1)

    wi_params = {
        "$select": "DateValue,ProjectSK,WorkItemId,WorkItemType,Title,State,StateCategory,StateChangeDate,Priority,TagNames,ParentWorkItemId,RevisedDate,AssignedToUserSK,ChangedByUserSK,CreatedByUserSK,OriginalEstimate,CompletedWork,RemainingWork,StoryPoints,WorkItemRevisionSK,AreaSK,IterationSK,ChangedDate,CreatedDate,ClosedDate,InProgressDate,CompletedDate",
        "$filter": f"DateValue ge {cur.isoformat()} and DateValue lt {next_day.isoformat()}",
        "$count": "true"
    }

    url = wi_url
    params = wi_params

    while url:
        resp = requests.get(url, auth=auth, params=params)
        resp.raise_for_status()
        data = resp.json()
        all_new_rows.extend(data["value"])

        url = data.get("@odata.nextLink")
        params = None  # nextLink already has query if present

    print(f"Loaded snapshot for {cur} (rows so far: {len(all_new_rows)})")
    cur = next_day

new_snapshots = pd.DataFrame(all_new_rows)
print("New rows loaded:", len(new_snapshots))

No new days to load – data is up to date.


SystemExit: 

C:\Users\ikolo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
wi_dates = [
    "DateValue", "ChangedDate", "CreatedDate",
    "ClosedDate", "InProgressDate", "CompletedDate", "StateChangeDate"
]

for col in wi_dates:
    new_snapshots[col] = pd.to_datetime(
        new_snapshots[col],
        utc=True,
        errors="coerce",     # bad/empty values -> NaT instead of crash
        format="ISO8601"     # handles with/without milliseconds
    )

new_snapshots = new_snapshots.rename(columns = {"DateValue": "SnapshotDate"})

In [ ]:
# Request for users
users_url = f"{base}/Users"
users_params = {
    # "$top": 10,
    "$select": "UserSK,UserName,UserEmail"
    }
resp_users = requests.get(users_url, auth=auth, params=users_params)
resp_users.raise_for_status()
users_df = pd.DataFrame(resp_users.json()["value"])

# Request for areas
areas_url = f"{base}/Areas"
areas_params = {
    # "$top": 10,
    "$select": "AreaSK,AreaName,AreaPath"
    }
resp_areas = requests.get(areas_url, auth=auth, params=areas_params)
resp_areas.raise_for_status()
areas_df = pd.DataFrame(resp_areas.json()["value"])

# Request for iterations
iters_url = f"{base}/Iterations"
iters_params = {
    "$select": "IterationSK,IterationName,IterationPath,StartDate,EndDate"
    }
resp_iter = requests.get(iters_url, auth=auth, params=iters_params)
resp_iter.raise_for_status()
iters_df = pd.DataFrame(resp_iter.json()["value"])
iters_dates = [
    "StartDate", "EndDate"
]

for date in iters_dates:
    iters_df[date] = pd.to_datetime(iters_df[date], utc=True)

# Request for projects
projects_url = f"{base}/Projects"
projects_params = {
    "$select": "ProjectSK,ProjectName"
    }
resp_projects = requests.get(projects_url, auth=auth, params=projects_params)
resp_projects.raise_for_status()
projects_df = pd.DataFrame(resp_projects.json()["value"])


In [ ]:
# take metadata from SK columns

user_sk_map = {
    "AssignedToUserSK": "AssignedTo",
    "ChangedByUserSK": "ChangedBy",
    "CreatedByUserSK": "CreatedBy"
}

new_snapshots = new_snapshots.copy()

for sk_col, name_col in user_sk_map.items():
    new_snapshots = (
        new_snapshots
        .merge(users_df[["UserSK", "UserName"]],
               left_on=sk_col, right_on="UserSK", how="left")
        .rename(columns={"UserName": name_col})
        .drop(columns=[sk_col, "UserSK"])
    )

new_snapshots = (
    new_snapshots
    .merge(areas_df, on="AreaSK", how="left")
    .drop(columns=["AreaSK"])
    .merge(iters_df, on="IterationSK", how="left")
    .rename(columns={"StartDate": "IterationStart", "EndDate": "IterationEnd"})
    .drop(columns=["IterationSK"])
    .merge(projects_df, on="ProjectSK", how="left")
    .drop(columns=["ProjectSK"])
)

new_snapshots["WorkItemURL"] = (
    base_url
    + "/"
    + new_snapshots["ProjectName"].astype(str)
    + "/_workitems/edit/"
    + new_snapshots["WorkItemId"].astype(str)
)

In [ ]:
# Write new data
work_items =  pd.concat([existing_snapshots, new_snapshots], ignore_index=True)

# Just in case
work_items = work_items.drop_duplicates(
    subset=["WorkItemId", "SnapshotDate"],
    keep="last"
)

work_items.to_parquet(snapshot_path, index=False)
print("Parquet updated. Total rows now:", len(work_items))

Parquet updated. Total rows now: 584823
